# Salary Master — Company Name & Alias Extraction (PySpark)

**Single-table pipeline:** reads the CSV from a Databricks Volume, extracts every identifiable primary / old / related company name, and writes **only one** Unity Catalog Delta table: `catalog.schema.salary_master`.

- **Preserve spelling and punctuation.** `&`, `PVT`, `LTD`, `Ltd.`, capitalization, and source misspellings remain unchanged. Only surrounding whitespace, known non-company annotations (currently `(DNS)`), relationship phrases, and unmatched *outer* wrapper brackets are removed.
- Stores aliases as multiple rows per Co Code; no extra raw or review table is created.
- `company_name_raw`, `parse_status`, and `needs_review` stay in **that same table** for auditing.
- `RELATED_ENTITY` indicates an association in the CSV; **it does not establish the same employer/legal entity**.
- The write mode is `overwrite` for a **complete replacement CSV**; adjust if your source is incremental.


In [ ]:
import re
from pyspark.sql import functions as F

# ===================== CHANGE THESE VALUES =====================
CATALOG = "your_catalog"
SCHEMA = "your_schema"
CSV_PATH = "/Volumes/your_catalog/your_schema/your_volume/company.csv"

CSV_CODE_COLUMN = "Co Code"
CSV_NAME_COLUMN = "Company Name"

SALARY_MASTER_TABLE = f"{CATALOG}.{SCHEMA}.salary_master"
WRITE_MODE = "overwrite"  # Full snapshot; replaces only salary_master
RUN_SELF_TEST = True
# ===============================================================

for identifier in (CATALOG, SCHEMA):
    if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", identifier):
        raise ValueError(f"Invalid catalog/schema identifier: {identifier!r}")

# Internal separator between extracted segments: must not appear in source data.
SEGMENT_SEPARATOR = "\u001e"

# Known NON-company annotation (extend only after validating source values).
ANNOTATION_REGEX = r"(?i)\(\s*DNS\s*\)"

# Case-insensitive relation phrases; rules recognize separators, not legal suffixes.
# Keep 'FORMER' ahead of generic 'KNOWN AS' to avoid misclassification.
ALIAS_RULES = [
    ("RELATED_ENTITY",
     r"(?i)(?:\s*[\(\[\{]+\s*)*\b(?:ALSO|ASLO)\s+INCLUDES?\b\s*:?\s*"),
    ("FORMER",
     r"(?i)(?:\s*[\(\[\{]+\s*)*\b(?:EARLIER|FORMERLY|PREVIOUSLY|WAS)"
     r"\s+(?:(?:KNOWN|KNOW)\s+AS|CALLED|NAMED)\b\s*:?\s*"),
    ("FORMER",
     r"(?i)(?:\s*[\(\[\{]+\s*)*\b(?:OLD|PREVIOUS)\s+NAME\b\s*[:\-]?\s*"),
    ("AKA",
     r"(?i)(?:\s*[\(\[\{]+\s*)*\b(?:ALSO\s+KNOWN\s+AS|KNOWN\s+AS|"
     r"KNOW\s+AS|TRADING\s+AS)\b\s*:?\s*"),
]

UNPARSED_RELATION_REGEX = (
    r"(?i)\b(?:ALSO|ASLO)\s+INCLUDES?\b|"
    r"\b(?:EARLIER|FORMERLY|PREVIOUSLY|WAS)\s+(?:KNOWN|KNOW)\s+AS\b|"
    r"\b(?:ALSO\s+KNOWN\s+AS|TRADING\s+AS|OLD\s+NAME)\b"
)

print("Output table:", SALARY_MASTER_TABLE)
print("Source CSV:", CSV_PATH)


In [ ]:
def count_char(column, regex_pattern):
    """Count one bracket character in a Spark column (built-in functions only)."""
    return F.length(column) - F.length(F.regexp_replace(column, regex_pattern, ""))


def extract_company_aliases(source_df):
    """Extract one row per discovered name while preserving source wording.

    Input columns: co_code, company_name_raw, source_record_id.
    All expressions are Spark-native; no Python UDFs / collecting raw data.
    """
    working = (
        source_df
        .withColumn("_marked", F.regexp_replace("company_name_raw", ANNOTATION_REGEX, " "))
        .withColumn("_marked", F.regexp_replace("_marked", r"[\r\n\t]", " "))
    )

    # Turn relationships into typed delimiters.
    for alias_type, pattern in ALIAS_RULES:
        working = working.withColumn(
            "_marked",
            F.regexp_replace("_marked", pattern, SEGMENT_SEPARATOR + alias_type + SEGMENT_SEPARATOR)
        )

    # E.g. [primary, FORMER, old_name, FORMER, older_name].
    segments = (
        working
        .withColumn("_parts", F.split("_marked", SEGMENT_SEPARATOR))
        .select("co_code", "company_name_raw", "source_record_id", "_parts",
                F.posexplode("_parts").alias("_pos", "_part"))
        .filter((F.col("_pos") % 2) == 0)
        .withColumn(
            "alias_type",
            F.when(F.col("_pos") == 0, F.lit("PRIMARY"))
             .otherwise(F.element_at(F.col("_parts"), F.col("_pos")))
        )
        .withColumn("_company", F.trim(F.regexp_replace("_part", r"\s+", " ")))
    )

    # Remove only *unmatched* outside brackets.
    # Example: "XYZ SOFTWARE PVT LTD)" -> "XYZ SOFTWARE PVT LTD".
    # Example: "ABC (INDIA) PVT LTD" stays unchanged.
    # At most one expression per bracket type / side, not a Python row UDF.
    bracket_specs = [
        (r"\(", r"\)", r"^(\(+)", r"(\)+)$"),
        (r"\[", r"\]", r"^(\[+)", r"(\]+)$"),
        (r"\{", r"\}", r"^(\{+)", r"(\}+)$"),
    ]
    for open_pattern, close_pattern, leading_pattern, trailing_pattern in bracket_specs:
        opening = count_char(F.col("_company"), open_pattern)
        closing = count_char(F.col("_company"), close_pattern)
        segments = segments.withColumn("_excess_close", F.greatest(closing - opening, F.lit(0)))
        trailing_count = F.length(F.regexp_extract("_company", trailing_pattern, 1))
        segments = segments.withColumn(
            "_company",
            F.when(
                (F.col("_excess_close") > 0) &
                (trailing_count >= F.col("_excess_close")),
                F.trim(F.expr("substring(_company, 1, length(_company) - _excess_close)"))
            ).otherwise(F.col("_company"))
        )

        opening = count_char(F.col("_company"), open_pattern)
        closing = count_char(F.col("_company"), close_pattern)
        segments = segments.withColumn("_excess_open", F.greatest(opening - closing, F.lit(0)))
        leading_count = F.length(F.regexp_extract("_company", leading_pattern, 1))
        segments = segments.withColumn(
            "_company",
            F.when(
                (F.col("_excess_open") > 0) &
                (leading_count >= F.col("_excess_open")),
                F.trim(F.expr("substring(_company, _excess_open + 1, length(_company))"))
            ).otherwise(F.col("_company"))
        )

    aliases = (
        segments
        .withColumn("company_name", F.trim("_company"))
        .filter(F.length("company_name") > 0)
    )

    # Review flags are columns of the *same* eventual salary_master table.
    unbalanced = (
        (count_char(F.col("company_name"), r"\(") != count_char(F.col("company_name"), r"\)")) |
        (count_char(F.col("company_name"), r"\[") != count_char(F.col("company_name"), r"\]")) |
        (count_char(F.col("company_name"), r"\{") != count_char(F.col("company_name"), r"\}"))
    )
    return (
        aliases
        .withColumn(
            "parse_issue",
            F.concat_ws(
                "; ",
                F.when(F.col("company_name").rlike(UNPARSED_RELATION_REGEX),
                       F.lit("UNPARSED_RELATION_PHRASE")),
                F.when(F.length("company_name") < 3, F.lit("VERY_SHORT_NAME")),
                F.when(unbalanced, F.lit("UNBALANCED_BRACKETS"))
            )
        )
        .withColumn("parse_status", F.when(F.length("parse_issue") > 0, "REVIEW")
                                      .otherwise("READY"))
        .select("co_code", "company_name", "alias_type", "parse_status",
                "parse_issue", "company_name_raw", "source_record_id")
        .dropDuplicates(["source_record_id", "company_name", "alias_type"])
    )


In [ ]:
if RUN_SELF_TEST:
    sample = [
        ("1001", "Arcum Engineering Ltd."),
        ("1002", "ABC & SONS PVT LTD (DNS)"),
        ("1003", "ADITYA BIRLA FINANCE LIMITED ASLO INCLUDES BIRLA INSURANCE "
                 "ADVISORY AND BROKING SERVICES LIMITED (( Earlier know as "
                 "BIRLA GLOBAL FINANCE LTD ( WAS KNOW AS BIRLA GLOBAL FINANCE COMPNAY LTD)"),
        ("1004", "XYZ TECHNOLOGIES LTD (Formerly known as XYZ SOFTWARE PVT LTD)"),
        ("1005", "ABC (INDIA) PVT LTD"),
        ("1006", "GLOBAL TECH LTD ALSO KNOWN AS GLOBAL TECHNOLOGIES"),
        ("1007", "ABC (INDIA) LTD (Earlier known as OLD ABC (INDIA) LTD)"),
    ]
    sample_df = (
        spark.createDataFrame(sample, ["co_code", "company_name_raw"])
        .withColumn("source_record_id", F.sha2(F.to_json(F.struct("co_code", "company_name_raw")), 256))
    )
    actual = {
        (row["co_code"], row["company_name"], row["alias_type"])
        for row in extract_company_aliases(sample_df)
            .select("co_code", "company_name", "alias_type").collect()
    }
    expected = {
        ("1001", "Arcum Engineering Ltd.", "PRIMARY"),
        ("1002", "ABC & SONS PVT LTD", "PRIMARY"),
        ("1003", "ADITYA BIRLA FINANCE LIMITED", "PRIMARY"),
        ("1003", "BIRLA INSURANCE ADVISORY AND BROKING SERVICES LIMITED", "RELATED_ENTITY"),
        ("1003", "BIRLA GLOBAL FINANCE LTD", "FORMER"),
        ("1003", "BIRLA GLOBAL FINANCE COMPNAY LTD", "FORMER"),
        ("1004", "XYZ TECHNOLOGIES LTD", "PRIMARY"),
        ("1004", "XYZ SOFTWARE PVT LTD", "FORMER"),
        ("1005", "ABC (INDIA) PVT LTD", "PRIMARY"),
        ("1006", "GLOBAL TECH LTD", "PRIMARY"),
        ("1006", "GLOBAL TECHNOLOGIES", "AKA"),
        ("1007", "ABC (INDIA) LTD", "PRIMARY"),
        ("1007", "OLD ABC (INDIA) LTD", "FORMER"),
    }
    assert actual == expected, (
        f"Extraction self-test failed! Missing={expected - actual}; Unexpected={actual - expected}"
    )
    print(f"Self-test passed: {len(expected)} sample aliases extracted exactly.")


In [ ]:
# ===================== READ CSV (NO RAW TABLE) =====================
csv_df = (
    spark.read.format("csv")
    .option("header", "true")
    .option("inferSchema", "false")  # Co Code remains a string (keeps leading zeros)
    .option("multiLine", "true")
    .option("quote", '"')
    .option("escape", '"')
    .option("mode", "FAILFAST")
    .load(CSV_PATH)
)

missing = [c for c in (CSV_CODE_COLUMN, CSV_NAME_COLUMN) if c not in csv_df.columns]
if missing:
    raise ValueError(f"Missing columns {missing}. Found: {csv_df.columns}")

source_df = (
    csv_df.select(
        F.trim(F.col(CSV_CODE_COLUMN).cast("string")).alias("co_code"),
        F.col(CSV_NAME_COLUMN).cast("string").alias("company_name_raw")
    )
    .dropDuplicates(["co_code", "company_name_raw"])
    .withColumn(
        "source_record_id",
        F.sha2(F.to_json(F.struct("co_code", "company_name_raw")), 256)
    )
)

if source_df.filter(F.col("company_name_raw").contains(SEGMENT_SEPARATOR)).limit(1).count():
    raise ValueError("Source contains an internal control character used for extraction.")

invalid_df = source_df.filter(
    F.col("co_code").isNull() |
    (F.length(F.col("co_code")) == 0) |
    F.col("company_name_raw").isNull() |
    (F.length(F.trim("company_name_raw")) == 0)
)
invalid_count = invalid_df.count()
if invalid_count:
    print(f"WARNING: {invalid_count} source records have empty Co Code or company text and will be skipped.")

valid_df = source_df.filter(
    F.col("co_code").isNotNull() &
    (F.length(F.col("co_code")) > 0) &
    F.col("company_name_raw").isNotNull() &
    (F.length(F.trim("company_name_raw")) > 0)
)

# ===================== EXTRACT ALIASES =====================
extracted_df = extract_company_aliases(valid_df)

# Consolidate duplicate aliases with deterministic source-reference selection.
# Only one row per (co_code, company_name, alias_type) in final salary_master.
consolidated_df = (
    extracted_df
    .groupBy("co_code", "company_name", "alias_type")
    .agg(
        F.max(F.when(F.col("parse_status") == "REVIEW", F.lit(1))
              .otherwise(F.lit(0))).alias("_has_issue"),
        F.concat_ws("; ", F.sort_array(F.collect_set(
            F.when(F.length("parse_issue") > 0, F.col("parse_issue"))
        ))).alias("parse_issue"),
        F.min("company_name_raw").alias("company_name_raw"),
        F.countDistinct("source_record_id").alias("source_record_count")
    )
    .withColumn("parse_status", F.when(F.col("_has_issue") > 0, "REVIEW")
                                 .otherwise("READY"))
    .drop("_has_issue")
)

# Identify names that appear under multiple company codes; don't silently choose one.
code_collisions = (
    consolidated_df.groupBy("company_name")
    .agg(F.countDistinct("co_code").alias("matched_co_code_count"))
)

salary_master_df = (
    consolidated_df
    .join(code_collisions, on="company_name", how="left")
    .withColumn(
        "needs_review",
        (F.col("parse_status") == "REVIEW") |
        (F.col("alias_type") == "RELATED_ENTITY") |
        (F.col("matched_co_code_count") > 1)
    )
    .withColumn("ingested_at", F.current_timestamp())
    .select(
        "co_code", "company_name", "alias_type",
        "parse_status", "parse_issue", "needs_review",
        "matched_co_code_count", "source_record_count",
        "company_name_raw", "ingested_at"
    )
)

if salary_master_df.limit(1).count() == 0:
    raise ValueError("No company names were extracted. salary_master was NOT overwritten.")

# ===================== WRITE ONLY ONE UC TABLE =====================
(
    salary_master_df.write
    .format("delta")
    .mode(WRITE_MODE)
    .option("overwriteSchema", "true")
    .saveAsTable(SALARY_MASTER_TABLE)
)

print("Persisted final salary master:", SALARY_MASTER_TABLE)
print("Total extracted company aliases:", spark.table(SALARY_MASTER_TABLE).count())

# Results / validation (no other table is written)
display(
    spark.table(SALARY_MASTER_TABLE)
    .groupBy("alias_type", "parse_status")
    .count()
    .orderBy("alias_type", "parse_status")
)

display(
    spark.table(SALARY_MASTER_TABLE)
    .select("co_code", "company_name", "alias_type", "needs_review")
    .orderBy("co_code", "company_name")
    .limit(100)
)


## Query the resulting table

```sql
SELECT co_code, company_name, alias_type, needs_review
FROM your_catalog.your_schema.salary_master
ORDER BY co_code, company_name;
```

**Data-quality notes:** `parse_status='READY'` only means that no configured parsing checks failed, not that a legal company identity was verified. Unrecognized description patterns may still require additional extraction rules. `needs_review=true` also flags related entities and company names associated with multiple Co Codes. The CSV is treated as a **full snapshot**, so each run overwrites `salary_master`.
